<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/Analisis_demanda_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# 1. CARGA DE DATOS
# =============================================================================
df_cliente_raw = pd.read_excel('TorreControlCobertura_16.07.xlsx')
df_erp_raw = pd.read_excel('Follow Up_Cat_26.07.xlsx')

# =============================================================================
# 2. ALINEACIÓN Y DETECCIÓN DINÁMICA DE SEMANAS
# =============================================================================
cols_estaticas_cliente = ['Part Number', 'Numero EPS', 'Class ID', 'Part Name', 'Facility ID', 'Rec Facility Name', 'Inv + Transit']

semanas_cliente_detectadas = sorted([col for col in df_cliente_raw.columns if col not in cols_estaticas_cliente], key=lambda x: int(x))
primera_semana_cliente = semanas_cliente_detectadas[0]

# Corrected logic to handle potential duplicate column names after renaming 'Back Order'
df_erp_temp = df_erp_raw.copy()
if primera_semana_cliente in df_erp_temp.columns and primera_semana_cliente != 'Back Order':
    # If the target week column already exists, and it's not the 'Back Order' itself, drop the existing one.
    # We assume 'Back Order' column has the correct data for the first week's ERP demand.
    df_erp_temp = df_erp_temp.drop(columns=[primera_semana_cliente])
df_erp_mapeado = df_erp_temp.rename(columns={'Back Order': primera_semana_cliente})

cols_estaticas_erp = ['Numero EPS']
semanas_erp_detectadas = [col for col in df_erp_mapeado.columns if col not in cols_estaticas_erp]

todas_las_semanas = sorted(list(set(semanas_cliente_detectadas) & set(semanas_erp_detectadas)), key=lambda x: int(x))

semanas_criticas = todas_las_semanas[:2]
semanas_futuras = todas_las_semanas[2:]
semanas_comercial = todas_las_semanas[:4]


# =============================================================================
# 3. PROCESAMIENTO Y CONSOLIDACIÓN (MERGE)
# =============================================================================
# Consolidación manteniendo la segregación por Facility ID
df_cliente = df_cliente_raw.groupby(['Numero EPS', 'Class ID', 'Facility ID'])[['Inv + Transit'] + todas_las_semanas].sum().reset_index()
df_erp = df_erp_mapeado.groupby(['Numero EPS'])[todas_las_semanas].sum().reset_index()

df_maestro = pd.merge(df_cliente, df_erp, on='Numero EPS', suffixes=('_CLIE', '_ERP'))


# =============================================================================
# 4. CÁLCULO EN CASCADA CON ACUMULACIÓN CRÍTICA (PLAN PLANTA)
# =============================================================================
df_maestro['Prioridad_Fabricacion'] = '🟢 Cubierto'
df_maestro['Semana_Faltante_Critico'] = 999
df_maestro['Cantidad_A_Producir_Urgente'] = 0

df_maestro['Faltante_Critico_Acumulado'] = (df_maestro[[f'{w}_CLIE' for w in semanas_criticas]].sum(axis=1) - df_maestro['Inv + Transit']).clip(lower=0)

inventario_operativo = df_maestro['Inv + Transit'].copy()

for indice, semana in enumerate(todas_las_semanas):
    nivel_prioridad = indice + 1
    col_cliente = f'{semana}_CLIE'

    demanda_semana = df_maestro[col_cliente]
    faltante_semana = (demanda_semana - inventario_operativo).clip(lower=0)
    inventario_operativo = (inventario_operativo - demanda_semana).clip(lower=0)

    filtro_primer_quiebre = (faltante_semana > 0) & (df_maestro['Semana_Faltante_Critico'] == 999)

    df_maestro.loc[filtro_primer_quiebre, 'Prioridad_Fabricacion'] = f'⭐ Prioridad {nivel_prioridad} (Semana {semana})'
    df_maestro.loc[filtro_primer_quiebre, 'Semana_Faltante_Critico'] = nivel_prioridad

    if indice < 2:
        df_maestro.loc[filtro_primer_quiebre, 'Cantidad_A_Producir_Urgente'] = df_maestro['Faltante_Critico_Acumulado']
    else:
        df_maestro.loc[filtro_primer_quiebre, 'Cantidad_A_Producir_Urgente'] = faltante_semana


# =============================================================================
# 5. CÁLCULOS PARA SISTEMA Y NEGOCIACIÓN COMERCIAL (AGREGADO DE COBERTURA EXACTA)
# =============================================================================
if semanas_futuras:
    df_maestro['Demanda_Futura_Cliente'] = df_maestro[[f'{w}_CLIE' for w in semanas_futuras]].sum(axis=1)
    df_maestro['Demanda_Futura_ERP'] = df_maestro[[f'{w}_ERP' for w in semanas_futuras]].sum(axis=1)
    df_maestro['Desalineacion_ERP'] = df_maestro['Demanda_Futura_ERP'] - df_maestro['Demanda_Futura_Cliente']
else:
    df_maestro['Demanda_Futura_Cliente'] = 0
    df_maestro['Demanda_Futura_ERP'] = 0
    df_maestro['Desalineacion_ERP'] = 0

df_maestro['Demanda_Cliente_4W'] = df_maestro[[f'{w}_CLIE' for w in semanas_comercial]].sum(axis=1)
df_maestro['Demanda_Critica_Cliente'] = df_maestro[[f'{w}_CLIE' for w in semanas_criticas]].sum(axis=1)

col_bo_mapeado = f'{primera_semana_cliente}_ERP'

# FUNCIÓN NUEVA: Calcula la semana máxima cubierta al 100% para cada Facility
def calcular_max_semana_cobertura(row, lista_semanas):
    inventario = row['Inv + Transit'] # Changed 'Inv' to 'Inv + Transit'
    if inventario <= 0:
        return "Sin Cobertura"

    ultima_semana_ok = "Ninguna"
    for sem in lista_semanas:
        demanda = row[f'{sem}_CLIE']
        if inventario >= demanda:
            inventario -= demanda
            ultima_semana_ok = f"Semana {sem}"
        else:
            break

    if ultima_semana_ok == f"Semana {lista_semanas[-1]}":
        return f"Total (> W{lista_semanas[-1]})"
    return ultima_semana_ok

# Aplicación del cálculo de cobertura exacta
df_maestro['Cobertura_Hasta_Semana'] = df_maestro.apply(lambda r: calcular_max_semana_cobertura(r, todas_las_semanas), axis=1)

def clasificar_alineacion_erp(row):
    umbral = 50
    if row['Desalineacion_ERP'] > umbral:
        return '⚠️ ERP Inflado (Riesgo Sobre-inventario)'
    elif row['Desalineacion_ERP'] < -umbral:
        return '🚨 ERP Corto (Riesgo Desabasto)'
    else:
        return '✅ ERP Alineado'

def bandera_negociacion(row):
    if (row[col_bo_mapeado] > 0) and (row['Inv + Transit'] >= row['Demanda_Cliente_4W']): # Changed 'Inv' to 'Inv + Transit'
        return f'🗣️ NEGOCIAR: Cancelar/Posponer BO W{primera_semana_cliente} (Cobertura >= 4 Semanas)'
    elif (row[col_bo_mapeado] > 0) and (row['Inv + Transit'] >= row['Demanda_Critica_Cliente']): # Changed 'Inv' to 'Inv + Transit'
        return f'⚠️ REVISAR: BO W{primera_semana_cliente} activo, pero corto plazo está cubierto'
    else:
        return 'Producción Normal'

df_maestro['Accion_ERP'] = df_maestro.apply(clasificar_alineacion_erp, axis=1)
df_maestro['Accion_Comercial'] = df_maestro.apply(bandera_negociacion, axis=1)


# =============================================================================
# 6. GENERACIÓN DE REPORTES FINALES
# =============================================================================
reporte_planta = df_maestro[df_maestro['Semana_Faltante_Critico'] != 999].copy()
reporte_planta = reporte_planta[['Class ID', 'Numero EPS', 'Inv + Transit', "Facility ID", 'Cantidad_A_Producir_Urgente', 'Prioridad_Fabricacion', 'Semana_Faltante_Critico']] # Changed 'Inv' to 'Inv + Transit'
reporte_planta = reporte_planta.sort_values(by=['Class ID', 'Semana_Faltante_Critico', 'Cantidad_A_Producir_Urgente'], ascending=[True, True, False])
reporte_planta = reporte_planta.drop(columns=['Semana_Faltante_Critico'])

# INCLUSIÓN EN REPORTE COMERCIAL: Se añade la columna 'Cobertura_Hasta_Semana'
reporte_negociacion = df_maestro[df_maestro['Accion_Comercial'].str.contains('NEGOCIAR') | df_maestro['Accion_Comercial'].str.contains('REVISAR')].copy()
nombre_col_bo = f'Back Order (W{primera_semana_cliente})'
reporte_negociacion = reporte_negociacion.rename(columns={col_bo_mapeado: nombre_col_bo})
reporte_negociacion = reporte_negociacion[['Numero EPS', 'Class ID', 'Facility ID', nombre_col_bo, 'Inv + Transit', 'Cobertura_Hasta_Semana', 'Accion_Comercial']]

reporte_sistema = df_maestro[['Class ID', "Facility ID", 'Numero EPS', 'Demanda_Futura_Cliente', 'Demanda_Futura_ERP', 'Desalineacion_ERP', 'Accion_ERP']]

# Mostrar en pantalla
print("="*75)
print(f"1. REPORTE DE PLANTA (PLAN_PLANTA CON VOLUMEN CRÍTICO ACUMULADO)")
print("="*75)
print(reporte_planta.to_string(index=False) if not reporte_planta.empty else "Todo cubierto. Sin requerimientos de fabricación.")

print("\n" + "="*75)
print(f"2. REPORTE DE NEGOCIACIÓN COMERCIAL")
print("="*75)
print(reporte_negociacion.to_string(index=False) if not reporte_negociacion.empty else "No hay oportunidades de negociación comercial.")

print("\n" + "="*75)
print(f"3. REPORTE DE AUDITORÍA ERP")
print("="*75)
print(reporte_sistema.to_string(index=False) if not reporte_sistema.empty else "No hay desalineaciones significativas en el ERP.")


1. REPORTE DE PLANTA (PLAN_PLANTA CON VOLUMEN CRÍTICO ACUMULADO)
Class ID                 Numero EPS  Inv + Transit Facility ID  Cantidad_A_Producir_Urgente      Prioridad_Fabricacion
      KB                390-7427-01              0          R2                            1  ⭐ Prioridad 9 (Semana 37)
      KB                337-4866-03              1          R2                            1 ⭐ Prioridad 11 (Semana 39)
    MASA                 1196336-01            -11          88                           23  ⭐ Prioridad 1 (Semana 29)
    MASA                 1196336-01              0          92                           10  ⭐ Prioridad 1 (Semana 29)
    MASA                 1196336-01              3          68                            1  ⭐ Prioridad 3 (Semana 31)
    MASA                 1196336-01             10          RS                           10 ⭐ Prioridad 15 (Semana 43)
     PTA                9M-7394-R06              0          13                          324  ⭐ Priorid

In [ ]:
# =============================================================================
# EXPORTACIÓN A EXCEL Y DESCARGA AUTOMÁTICA
# =============================================================================
from google.colab import files

# Definir el nombre del archivo de salida
nombre_archivo = 'Analisis_Prioridades_y_Cobertura_16.07.xlsx'

print("Generando archivo de Excel...")

# Guardar cada reporte en una pestaña diferente
with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:
    reporte_planta.to_excel(writer, sheet_name='Plan_Planta', index=False)
    reporte_negociacion.to_excel(writer, sheet_name='Negociacion_Comercial', index=False)
    reporte_sistema.to_excel(writer, sheet_name='Auditoria_ERP', index=False)
    df_maestro.to_excel(writer, sheet_name='Data_Maestra_Completa', index=False)

print(f"¡Archivo '{nombre_archivo}' creado con éxito!")
print("Iniciando descarga automática...")

# Descargar el archivo directamente a tu equipo local
files.download(nombre_archivo)

Generando archivo de Excel...
¡Archivo 'Analisis_Prioridades_y_Cobertura_16.07.xlsx' creado con éxito!
Iniciando descarga automática...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>